# EDA — Chatbot NLP de PQRS en Salud con Español Simplificado y Wayuunaiki Básico

**Proyecto de aula — Procesamiento de Lenguaje Natural (NLP)**

Este notebook realiza el **Análisis Exploratorio de Datos (EDA)** del corpus de
mensajes dirigidos a un chatbot de **Peticiones, Quejas, Reclamos y Sugerencias
(PQRS)** en el dominio de la salud, con soporte para:

- **Lenguaje formal**: el típico de trámites administrativos en salud.
- **Lenguaje simplificado**: oraciones cortas y claras (accesibilidad cognitiva).
- **Wayuunaiki básico**: mensajes con **inserción léxica real** (*code-switching*)
  de palabras y frases en wayuunaiki, obtenidas directamente del diccionario
  bilingüe público del repositorio
  [`Manuel-2011/RLTranslator`](https://github.com/Manuel-2011/RLTranslator/tree/main/datasets),
  combinadas con estructura en español para los términos institucionales que no
  tienen traducción directa (EPS, autorización, reclamo, etc.).

**Estructura del notebook:**
1. Carga e integración de datos (léxico real de wayuunaiki + dataset sintético de PQRS)
2. Carga y descripción general del dataset
3. Estadísticas descriptivas y visualizaciones léxicas/semánticas
4. Identificación de problemas de calidad de datos

> **Nota de honestidad académica:** el módulo en wayuunaiki no genera gramática
> completa del idioma (tarea de traducción automática de alta complejidad, fuera
> del alcance de este EDA). En su lugar, inserta **palabras y frases reales**
> extraídas del diccionario del repositorio referenciado dentro de una estructura
> de oración en español — un patrón de *code-switching* léxico, consistente con
> cómo se documentan comunidades bilingües de bajos recursos en la literatura de NLP.


## 0. Librerías y configuración

Se importan las librerías estándar para manipulación de datos (`pandas`, `numpy`),
descarga de datos (`urllib`/`requests`), visualización (`matplotlib`, `seaborn`,
`wordcloud`) y procesamiento de texto (`sklearn`). Se fija una semilla aleatoria
para la reproducibilidad del dataset sintético.

In [ ]:
import io
import re
import random
import urllib.request
import urllib.error

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from wordcloud import WordCloud

# Configuración de estilo para las visualizaciones
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Semilla para reproducibilidad
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_colwidth", 120)


## 1. Carga e Integración de Datos (Sintéticos + Léxico Real de Wayuunaiki)

### 1.1 Descarga del diccionario real `dictionary.csv`

Se descarga directamente el diccionario bilingüe español–wayuunaiki
(`spa`, `guc`) publicado en el repositorio público
[`Manuel-2011/RLTranslator`](https://github.com/Manuel-2011/RLTranslator/tree/main/datasets),
asociado al artículo *"Improving Low-Resource Translation with Dictionary-Guided
Fine-Tuning and RL: A Spanish–Wayuunaiki Study"*. Dicho diccionario contiene más
de 74.000 pares de traducción y alimenta el módulo de expresiones en lengua
nativa del chatbot.

Si la descarga falla (sin conexión a internet, límite de tasa de GitHub, etc.),
se utiliza un **diccionario de respaldo embebido** con una pequeña muestra de
pares verificados, para que el notebook siga siendo completamente ejecutable
(requisito importante para un entorno como Google Colab).

In [ ]:
DICTIONARY_URL = (
    "https://raw.githubusercontent.com/Manuel-2011/RLTranslator/main/"
    "datasets/dictionary.csv"
)

# Diccionario de respaldo (offline) por si falla la descarga del recurso real.
# Son pares reales tomados de una muestra previa del mismo `dictionary.csv`.
DICCIONARIO_RESPALDO_CSV = """spa,guc
yo,taya
hola,jamaan
hola,jamaya
por favor,jalia tamüin
gracias,analu'ulu pümüin
dolor,aishatasü
enfermo,ayuulii
enfermedad,wanülü
medico,tottoolu
ayuda,akaaliijaa
fiebre,tuttaa
sangre,asha
tengo una cita,antiraainjachi taya
pida cita,puchunta wanee cita
estoy enfermo,ayuuishi taya
estoy saludable,anashi taya
"""

def cargar_diccionario_wayuunaiki(url=DICTIONARY_URL, timeout=15):
    """Descarga el diccionario real spa-guc desde RLTranslator. Si falla,
    recurre al respaldo embebido. Devuelve (DataFrame, fuente)."""
    try:
        with urllib.request.urlopen(url, timeout=timeout) as respuesta:
            contenido = respuesta.read().decode("utf-8")
        df_dic = pd.read_csv(io.StringIO(contenido))
        fuente = "descarga_real_github"
    except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, Exception) as e:
        print(f"[Aviso] No fue posible descargar el diccionario real ({e}).")
        print("        Se usará el diccionario de respaldo embebido.")
        df_dic = pd.read_csv(io.StringIO(DICCIONARIO_RESPALDO_CSV))
        fuente = "respaldo_offline"

    df_dic.columns = [c.strip().lower() for c in df_dic.columns]
    df_dic = df_dic.rename(columns={df_dic.columns[0]: "spa", df_dic.columns[1]: "guc"})
    df_dic["spa"] = df_dic["spa"].astype(str).str.strip().str.lower()
    df_dic["guc"] = df_dic["guc"].astype(str).str.strip()
    df_dic = df_dic.dropna(subset=["spa", "guc"]).drop_duplicates()
    return df_dic, fuente

df_diccionario, fuente_diccionario = cargar_diccionario_wayuunaiki()

print(f"Fuente utilizada: {fuente_diccionario}")
print(f"Pares español-wayuunaiki cargados: {df_diccionario.shape[0]}")
df_diccionario.head(10)


### 1.2 Construcción de un glosario de salud a partir del léxico real

No todo el vocabulario institucional de salud (EPS, autorización, reclamo,
medicamento como concepto general, etc.) tiene traducción directa en un
diccionario general como este — es un hallazgo esperable en lenguas de bajos
recursos. Por ello, se construye una función de búsqueda que:

1. Busca primero una coincidencia **exacta** del término en español.
2. Si no existe, busca coincidencias **parciales** y prioriza traducciones
   cortas (1–2 palabras) para evitar frases largas y ruidosas.
3. Si no hay ninguna coincidencia limpia, el término queda documentado como
   una **brecha léxica** (y se mantendrá en español dentro de los mensajes,
   reflejando el *code-switching* real que ocurre en estos contextos).

In [ ]:
TERMINOS_CLAVE_SALUD = [
    "yo", "hola", "por favor", "gracias", "dolor", "enfermo", "enfermedad",
    "medico", "ayuda", "fiebre", "sangre", "tengo una cita",
    "medicamento", "hospital", "paciente", "consulta", "autorizacion",
]

def buscar_glosa(termino, df_dic):
    """Busca la mejor traducción disponible para `termino` en el diccionario
    real. Devuelve (traduccion_o_None, tipo_de_match)."""
    exacta = df_dic[df_dic["spa"] == termino]
    if len(exacta):
        return exacta.iloc[0]["guc"], "exacta"

    patron = rf"\b{re.escape(termino)}\w*\b"
    parcial = df_dic[df_dic["spa"].str.contains(patron, regex=True, na=False)].copy()
    if len(parcial) == 0:
        return None, "sin_match"

    parcial["n_palabras_guc"] = parcial["guc"].str.split().apply(len)
    parcial["n_palabras_spa"] = parcial["spa"].str.split().apply(len)
    candidatos = parcial[parcial["n_palabras_guc"] <= 2].sort_values(
        ["n_palabras_spa", "n_palabras_guc"]
    )
    if len(candidatos):
        return candidatos.iloc[0]["guc"], "aproximada"
    return None, "sin_match_limpio"

glosario_filas = []
for termino in TERMINOS_CLAVE_SALUD:
    traduccion, tipo_match = buscar_glosa(termino, df_diccionario)
    glosario_filas.append({"termino_es": termino, "traduccion_guc": traduccion, "tipo_match": tipo_match})

glosario_salud = pd.DataFrame(glosario_filas)
glosario_salud


In [ ]:
# Diccionario de acceso rápido {término_es: traducción_guc} solo con términos
# que sí tienen una traducción utilizable (exacta o aproximada y corta).
GLOSARIO = {
    fila["termino_es"]: fila["traduccion_guc"]
    for fila in glosario_filas
    if fila["traduccion_guc"] is not None
}

cobertura = glosario_salud["tipo_match"].value_counts()
print("Cobertura léxica del diccionario real para vocabulario de salud/PQRS:")
print(cobertura)
print(f"\nTérminos con traducción utilizable: {len(GLOSARIO)} de {len(TERMINOS_CLAVE_SALUD)}")

plt.figure(figsize=(7, 4))
sns.barplot(x=cobertura.index, y=cobertura.values, hue=cobertura.index,
            palette="crest", legend=False)
plt.title("Cobertura del diccionario real RLTranslator para vocabulario de salud/PQRS")
plt.ylabel("Número de términos")
plt.xlabel("Tipo de coincidencia")
plt.tight_layout()
plt.show()


### 1.3 (Opcional) Exploración del corpus paralelo `train.es` / `train.guc`

El repositorio también incluye corpus paralelos oración-a-oración
(`train.es`, `train.guc`, usados en el *AmericasNLP Shared Task*). Se descargan
de forma opcional para ilustrar frases completas reales en contexto de salud,
que sirven como referencia de estilo para los mensajes simulados (no se integran
literalmente al dataset de PQRS, dado que son oraciones genéricas de un corpus
de traducción general y no específicas del dominio administrativo en salud).

In [ ]:
TRAIN_ES_URL = "https://raw.githubusercontent.com/Manuel-2011/RLTranslator/main/datasets/train.es"
TRAIN_GUC_URL = "https://raw.githubusercontent.com/Manuel-2011/RLTranslator/main/datasets/train.guc"

PALABRAS_CLAVE_SALUD = ["dolor", "enferm", "hospital", "medic", "fiebre", "salud", "paciente"]

def cargar_lineas(url, timeout=15):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as respuesta:
            return respuesta.read().decode("utf-8").splitlines()
    except Exception as e:
        print(f"[Aviso] No fue posible descargar {url} ({e}). Se omite esta exploración opcional.")
        return []

lineas_es = cargar_lineas(TRAIN_ES_URL)
lineas_guc = cargar_lineas(TRAIN_GUC_URL)

if lineas_es and lineas_guc and len(lineas_es) == len(lineas_guc):
    print(f"Oraciones paralelas cargadas: {len(lineas_es)}")
    idx_salud = [i for i, linea in enumerate(lineas_es)
                 if any(p in linea.lower() for p in PALABRAS_CLAVE_SALUD)]
    print(f"Oraciones relacionadas con salud encontradas: {len(idx_salud)}")

    muestra = pd.DataFrame({
        "es": [lineas_es[i] for i in idx_salud[:8]],
        "guc": [lineas_guc[i] for i in idx_salud[:8]],
    })
    display(muestra)
else:
    print("Exploración opcional de train.es/train.guc omitida (sin datos disponibles).")


### 1.4 Plantillas de mensajes sintéticos por intención y nivel de lenguaje

Se definen plantillas parametrizadas para los niveles `formal` y `simplificado`
(español estándar) y para `wayuunaiki_basico`, donde las palabras clave
(`{yo}`, `{saludo}`, `{dolor}`, `{medico}`, etc.) se **sustituyen por las
traducciones reales** obtenidas en `GLOSARIO`. Los términos institucionales sin
traducción disponible (EPS, autorización, nombre del medicamento) se mantienen
en español, reproduciendo el *code-switching* real de estos contextos.

In [ ]:
NOMBRES = ["María", "José", "Carmen", "Luis", "Ana", "Pedro", "Rosa", "Jairo",
           "Yolanda", "Kaliana", "Jimena", "Camilo", "Estefany", "Doris", "Nelson"]

MEDICAMENTOS = ["Losartán", "Metformina", "Acetaminofén", "Insulina", "Omeprazol",
                "Enalapril", "Amoxicilina", "Salbutamol", "Warfarina", "Ibuprofeno"]

EPS = ["SaludTotal", "ViveEPS", "MediSalud", "EPS Cesar Salud", "BienestarPlus"]

ESPECIALIDADES = ["medicina general", "pediatría", "ginecología", "cardiología",
                   "odontología", "oftalmología", "psicología"]

CIUDADES = ["Valledupar", "Riohacha", "Maicao", "Uribia", "Manaure", "Fonseca"]

def fecha_aleatoria():
    dia = random.randint(1, 28)
    mes = random.randint(1, 12)
    anio = random.choice([2024, 2025])
    return f"{dia:02d}/{mes:02d}/{anio}"

# Solo estas claves se etiquetan como entidades del dominio PQRS;
# las claves del GLOSARIO (saludo, yo, dolor, etc.) son léxico funcional,
# no entidades nombradas.
PLACEHOLDER_TAGS = {
    "nombre": "PACIENTE",
    "medicamento": "MEDICAMENTO",
    "eps": "EPS",
    "fecha": "FECHA",
    "especialidad": "ESPECIALIDAD",
    "ciudad": "CIUDAD",
}

TEMPLATES = {
    "solicitar_cita": {
        "formal": [
            "Buenos días, deseo solicitar una cita con {especialidad} para el paciente "
            "{nombre}, afiliado a {eps}, preferiblemente antes del {fecha}.",
            "Solicito formalmente la asignación de cita médica en {especialidad} en la "
            "sede de {ciudad}; mi afiliación corresponde a la EPS {eps}.",
            "Por medio del presente mensaje solicito agendar cita de {especialidad} para "
            "{nombre}, afiliado a {eps}, en la ciudad de {ciudad}.",
        ],
        "simplificado": [
            "Hola, necesito una cita con el médico de {especialidad} para {nombre}.",
            "Quiero pedir cita en {ciudad}, es para {especialidad}, por favor.",
            "Necesito cita pronto, es para {nombre}, con {especialidad}.",
        ],
        "wayuunaiki_basico": [
            "{saludo}, {yo} {nombre}. Necesito {cita_t} con {medico_t} de {especialidad}, "
            "EPS {eps}.",
            "{saludo}, {yo} {nombre}. {por_favor_t}, una cita de {especialidad} en "
            "{ciudad}, EPS {eps}.",
        ],
    },
    "queja_atencion": {
        "formal": [
            "Por medio de la presente formulo una queja formal respecto a la atención "
            "recibida el día {fecha} en la sede de {ciudad} por parte de personal de {eps}.",
            "Deseo dejar constancia de mi inconformidad con el trato recibido por parte "
            "del personal administrativo de {eps} el día {fecha}.",
            "Presento queja formal por la deficiente atención prestada a {nombre} en "
            "{ciudad} por parte de {eps}.",
        ],
        "simplificado": [
            "Quiero quejarme, me trataron mal en la EPS {eps} el día {fecha}.",
            "No me gustó cómo me atendieron en {ciudad}, fue muy grosero el trato.",
            "Me atendieron muy mal en {eps}, quiero poner una queja.",
        ],
        "wayuunaiki_basico": [
            "{saludo}, {yo} {nombre}. Mala atención en EPS {eps} el {fecha}. {ayuda_t}, "
            "por favor.",
            "{saludo}, {yo} {nombre}. No me gustó la atención en {ciudad}, EPS {eps}, "
            "el {fecha}. {dolor_t}.",
        ],
    },
    "reclamo_medicamentos": {
        "formal": [
            "Formulo reclamo por la no entrega oportuna del medicamento {medicamento} "
            "formulado para el paciente {nombre}, afiliado a {eps}.",
            "Solicito la entrega inmediata de {medicamento}, el cual no ha sido "
            "dispensado desde el {fecha} por la EPS {eps}.",
            "Presento reclamo formal ante {eps} por la falta de suministro de "
            "{medicamento} al paciente {nombre}.",
        ],
        "simplificado": [
            "No me han dado el medicamento {medicamento}, lo necesito ya.",
            "Hace días pedí {medicamento} y en {eps} todavía no me lo entregan.",
            "Necesito que me entreguen {medicamento}, ya llevo esperando desde {fecha}.",
        ],
        "wayuunaiki_basico": [
            "{saludo}, {yo} {nombre}. {enfermo_t}, necesito el medicamento {medicamento}, "
            "EPS {eps}, desde {fecha}.",
            "{saludo}, {yo} {nombre}. {dolor_t} por {enfermedad_t}, no me dan "
            "{medicamento} en EPS {eps}.",
        ],
    },
    "consultar_autorizacion": {
        "formal": [
            "Solicito información sobre el estado de la autorización del procedimiento "
            "médico requerido por el paciente {nombre} ante {eps}.",
            "Requiero conocer el estado actual de la autorización de {especialidad} "
            "tramitada el {fecha} con la EPS {eps}.",
            "Agradezco informar el estado de la autorización solicitada por {nombre} "
            "en la ciudad de {ciudad}.",
        ],
        "simplificado": [
            "¿Cómo va mi autorización para {especialidad}? La pedí en {eps}.",
            "Quiero saber si ya está lista mi autorización, {nombre} es el paciente.",
            "¿Ya salió la autorización que pedí el {fecha} en {eps}?",
        ],
        "wayuunaiki_basico": [
            "{saludo}, {yo} {nombre}. {por_favor_t}, ¿autorización de {especialidad}, "
            "EPS {eps}?",
            "{saludo}, {yo} {nombre}. {gracias_t}, ¿está lista mi autorización de "
            "{especialidad} en EPS {eps}?",
        ],
    },
}


In [ ]:
# -----------------------------------------------------------------------------
# Generador de registros: combina entidades sintéticas + léxico real (GLOSARIO)
# -----------------------------------------------------------------------------
def valor_glosario(clave_glosario, por_defecto=""):
    """Devuelve la traducción real si existe en GLOSARIO; si no, un
    marcador en español (brecha léxica documentada en la Sección 1.2)."""
    return GLOSARIO.get(clave_glosario, por_defecto)

def generar_valores():
    return {
        "nombre": random.choice(NOMBRES),
        "medicamento": random.choice(MEDICAMENTOS),
        "eps": random.choice(EPS),
        "fecha": fecha_aleatoria(),
        "especialidad": random.choice(ESPECIALIDADES),
        "ciudad": random.choice(CIUDADES),
        # Léxico real tomado del diccionario RLTranslator (ver GLOSARIO):
        "saludo": random.choice([valor_glosario("hola", "Hola"), valor_glosario("hola", "Hola")]),
        "yo": valor_glosario("yo", "yo"),
        "por_favor_t": valor_glosario("por favor", "por favor"),
        "gracias_t": valor_glosario("gracias", "gracias"),
        "dolor_t": valor_glosario("dolor", "dolor"),
        "enfermo_t": valor_glosario("enfermo", "enfermo"),
        "enfermedad_t": valor_glosario("enfermedad", "enfermedad"),
        "medico_t": valor_glosario("medico", "médico"),
        "ayuda_t": valor_glosario("ayuda", "ayuda"),
        "cita_t": valor_glosario("tengo una cita", "cita"),
    }

def construir_registro(intencion, nivel):
    """Construye un mensaje sintético y sus entidades etiquetadas a partir
    de una plantilla aleatoria para la combinación (intencion, nivel)."""
    template = random.choice(TEMPLATES[intencion][nivel])
    valores = generar_valores()
    texto = template.format(**valores)

    placeholders_usados = re.findall(r"\{(\w+)\}", template)
    entidades = "; ".join(
        f"{PLACEHOLDER_TAGS[p]}:{valores[p]}"
        for p in placeholders_usados if p in PLACEHOLDER_TAGS
    )
    if entidades == "":
        entidades = "SIN_ENTIDADES"
    return texto, entidades

N_REGISTROS = 220
INTENCIONES = ["solicitar_cita", "queja_atencion", "reclamo_medicamentos", "consultar_autorizacion"]
NIVELES = ["formal", "simplificado", "wayuunaiki_basico"]

# Distribución intencionalmente desbalanceada para simular un problema real
# de calidad de datos (ver Sección 4 - Desbalance de clases).
PESOS_INTENCION = [0.40, 0.15, 0.30, 0.15]   # solicitar_cita sobre-representada
PESOS_NIVEL = [0.55, 0.30, 0.15]             # wayuunaiki_basico sub-representado

registros = []
for _ in range(N_REGISTROS):
    intencion = random.choices(INTENCIONES, weights=PESOS_INTENCION, k=1)[0]
    nivel = random.choices(NIVELES, weights=PESOS_NIVEL, k=1)[0]
    texto, entidades = construir_registro(intencion, nivel)
    registros.append({
        "texto_usuario": texto,
        "intencion": intencion,
        "nivel_lenguaje": nivel,
        "entidades_etiquetadas": entidades,
    })

df = pd.DataFrame(registros)
print(f"Registros generados: {df.shape[0]}")
df[df["nivel_lenguaje"] == "wayuunaiki_basico"].head(6)


### 1.5 Inyección controlada de imperfecciones (ruido, duplicados, nulos)

Para que el EDA de la Sección 4 tenga problemas reales de calidad que detectar, se
inyectan de forma controlada: (a) **ruido ortográfico/gramatical** simulado en una
fracción de los mensajes, (b) **registros duplicados** exactos, y (c) **valores
nulos y vacíos**, como podrían producirse por fallos de captura del chatbot.

In [ ]:
def introducir_ruido(texto):
    """Simula errores ortográficos/gramaticales comunes: elimina tildes y
    duplica una letra al azar dentro de una palabra."""
    mapa_tildes = {"á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u"}
    texto_ruido = "".join(mapa_tildes.get(c, c) for c in texto)

    palabras = texto_ruido.split()
    if len(palabras) > 3:
        idx = random.randint(0, len(palabras) - 1)
        palabra = palabras[idx]
        if len(palabra) > 1:
            palabras[idx] = palabra[0] * 2 + palabra[1:]
        texto_ruido = " ".join(palabras)
    return texto_ruido

# Se aplica ruido ortográfico al 12% de los registros
idx_ruido = df.sample(frac=0.12, random_state=RANDOM_SEED).index
df.loc[idx_ruido, "texto_usuario"] = df.loc[idx_ruido, "texto_usuario"].apply(introducir_ruido)

# Se generan 8 duplicados exactos (copiando registros existentes)
duplicados = df.sample(n=8, random_state=RANDOM_SEED)
df = pd.concat([df, duplicados], ignore_index=True)

# Se insertan valores nulos (NaN) simulando fallos de captura del chatbot
idx_nulos = df.sample(n=6, random_state=RANDOM_SEED + 1).index
df.loc[idx_nulos, "texto_usuario"] = np.nan

# Se insertan cadenas vacías, otro tipo común de "vacío" en datos reales
idx_vacios = df.sample(n=5, random_state=RANDOM_SEED + 2).index
df.loc[idx_vacios, "texto_usuario"] = ""

print(f"Dataset final: {df.shape[0]} registros, {df.shape[1]} columnas")
df.sample(5, random_state=RANDOM_SEED)


## 2. Carga y Descripción General del Dataset

Se inspecciona la estructura general del `DataFrame`: forma, tipos de datos,
una vista preliminar de los registros y estadísticas descriptivas generales.

In [ ]:
print("Forma del dataset (filas, columnas):", df.shape)
print("\nTipos de datos por columna:")
print(df.dtypes)
print("\nInformación general del DataFrame:")
df.info()


In [ ]:
df.head(10)


In [ ]:
df.describe(include="all")


## 3. Estadísticas Descriptivas y Visualizaciones (Léxicas y Semánticas)

### 3.1 Métricas de longitud

Se calculan dos métricas básicas por mensaje: número de caracteres
(`len(texto)`) y número de palabras (`len(texto.split())`). Estas métricas
ayudan a caracterizar la complejidad textual por nivel de lenguaje e intención.

In [ ]:
# Se calcula sobre el texto tal cual; los nulos se tratan como cadena vacía
# únicamente para el cálculo de longitud (no se modifica la columna original).
df["longitud_char"] = df["texto_usuario"].fillna("").apply(len)
df["conteo_palabras"] = df["texto_usuario"].fillna("").apply(lambda t: len(t.split()))

df[["longitud_char", "conteo_palabras"]].describe()


### 3.2 Distribución de longitud de mensaje por nivel de lenguaje

Se visualiza la distribución de `longitud_char` y `conteo_palabras`, segmentada
por `nivel_lenguaje`, mediante histogramas con estimación de densidad (KDE).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x="longitud_char", hue="nivel_lenguaje", kde=True,
             element="step", ax=axes[0])
axes[0].set_title("Longitud de mensaje (caracteres) por nivel de lenguaje")
axes[0].set_xlabel("Número de caracteres")

sns.histplot(data=df, x="conteo_palabras", hue="nivel_lenguaje", kde=True,
             element="step", ax=axes[1])
axes[1].set_title("Longitud de mensaje (palabras) por nivel de lenguaje")
axes[1].set_xlabel("Número de palabras")

plt.tight_layout()
plt.show()


### 3.3 Frecuencia de términos: N-gramas, TF-IDF y nubes de palabras

Se identifican los **unigramas y bigramas** más frecuentes por cada `intencion`
(usando `TfidfVectorizer`), el ranking global de términos por **TF-IDF promedio**
y se generan **nubes de palabras** por `nivel_lenguaje` para una inspección
semántica rápida.

Se define una lista básica de *stopwords* en español (incluyendo fórmulas de
cortesía frecuentes en el dominio) para filtrar palabras poco informativas.

In [ ]:
SPANISH_STOPWORDS = [
    "de", "la", "que", "el", "en", "y", "a", "los", "del", "se", "las", "por",
    "un", "para", "con", "no", "una", "su", "al", "es", "lo", "como", "más",
    "mi", "ya", "me", "si", "porque", "esta", "entre", "cuando", "muy", "sin",
    "sobre", "también", "hay", "donde", "quien", "desde", "todo", "nos",
    "durante", "todos", "uno", "les", "ni", "contra", "otros", "ese", "eso",
    "ante", "ellos", "e", "esto", "mí", "antes", "algunos", "qué", "unos",
    "yo", "otro", "otras", "otra", "él", "tanto", "esa", "estos", "mucho",
    "quienes", "nada", "muchos", "cual", "sea", "poco", "ella", "estar",
    "haber", "estas", "estaba", "hola", "buenos", "días", "favor", "quiero",
    "deseo", "presente", "medio", "día",
]

# Se descartan registros con texto nulo o vacío únicamente para el análisis léxico
texto_valido = df.dropna(subset=["texto_usuario"]).copy()
texto_valido = texto_valido[texto_valido["texto_usuario"].str.strip() != ""]

print(f"Mensajes válidos para análisis léxico: {texto_valido.shape[0]} de {df.shape[0]}")


In [ ]:
# Top unigramas/bigramas por intención (TfidfVectorizer, ordenado por suma de TF-IDF)
for intencion in texto_valido["intencion"].unique():
    subset = texto_valido.loc[texto_valido["intencion"] == intencion, "texto_usuario"]

    vectorizer = TfidfVectorizer(stop_words=SPANISH_STOPWORDS, ngram_range=(1, 2), max_features=20)
    matriz = vectorizer.fit_transform(subset)
    puntajes = matriz.sum(axis=0).A1
    terminos = vectorizer.get_feature_names_out()
    top_terminos = sorted(zip(terminos, puntajes), key=lambda x: x[1], reverse=True)[:10]

    print(f"\nTop términos/bigramas (TF-IDF) — intención: {intencion}")
    for termino, score in top_terminos:
        print(f"  {termino:<25} {score:.3f}")


In [ ]:
# Ranking global de términos por TF-IDF promedio (dataset completo)
tfidf_vectorizer = TfidfVectorizer(stop_words=SPANISH_STOPWORDS, ngram_range=(1, 2), max_features=30)
tfidf_matriz = tfidf_vectorizer.fit_transform(texto_valido["texto_usuario"])
tfidf_promedio = tfidf_matriz.mean(axis=0).A1
tfidf_terminos = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame({"termino": tfidf_terminos, "tfidf_promedio": tfidf_promedio})
tfidf_df = tfidf_df.sort_values("tfidf_promedio", ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=tfidf_df, x="tfidf_promedio", y="termino", hue="termino",
            palette="viridis", legend=False)
plt.title("Top 20 términos por puntaje TF-IDF promedio (dataset completo)")
plt.xlabel("TF-IDF promedio")
plt.ylabel("Término / bigrama")
plt.tight_layout()
plt.show()


In [ ]:
# Nubes de palabras por nivel de lenguaje
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, nivel in zip(axes, NIVELES):
    texto_nivel = " ".join(texto_valido.loc[texto_valido["nivel_lenguaje"] == nivel, "texto_usuario"])
    wc = WordCloud(width=500, height=400, background_color="white",
                   stopwords=set(SPANISH_STOPWORDS), collocations=False).generate(texto_nivel)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"Nube de palabras — nivel: {nivel}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 4. Identificación de Problemas de Calidad de Datos

### 4.1 Desbalance de clases

Se visualiza la distribución de frecuencias por `intencion` y por
`nivel_lenguaje` para evidenciar el desbalance introducido en la generación del
dataset (relevante para decisiones de *sampling*/ponderación en el entrenamiento
del clasificador de intenciones).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x="intencion", hue="intencion",
              order=df["intencion"].value_counts().index,
              palette="mako", legend=False, ax=axes[0])
axes[0].set_title("Distribución de frecuencias por intención")
axes[0].tick_params(axis="x", rotation=25)

sns.countplot(data=df, x="nivel_lenguaje", hue="nivel_lenguaje",
              order=df["nivel_lenguaje"].value_counts().index,
              palette="rocket", legend=False, ax=axes[1])
axes[1].set_title("Distribución de frecuencias por nivel de lenguaje")

plt.tight_layout()
plt.show()

print("Conteo por intención:")
print(df["intencion"].value_counts())
print("\nConteo por nivel de lenguaje:")
print(df["nivel_lenguaje"].value_counts())


### 4.2 Valores nulos y duplicados: identificación, reporte y limpieza

Se identifican explícitamente los registros con `texto_usuario` nulo o vacío,
así como las filas duplicadas (exactas y por texto), y se construye una versión
**limpia** del dataset (`df_limpio`) sin alterar el `DataFrame` original.

In [ ]:
# --- Valores nulos ---
print("Valores nulos por columna:")
print(df.isnull().sum())

n_vacios = (df["texto_usuario"].fillna("").str.strip() == "").sum()
print(f"\nRegistros con texto vacío (''): {n_vacios}")

# --- Duplicados ---
n_dup_totales = df.duplicated().sum()
n_dup_texto = df.duplicated(subset=["texto_usuario"]).sum()
print(f"\nFilas completamente duplicadas: {n_dup_totales}")
print(f"Duplicados considerando solo 'texto_usuario': {n_dup_texto}")

df[df.duplicated(subset=["texto_usuario"], keep=False) & df["texto_usuario"].notna()] \
    .sort_values("texto_usuario").head(6)


In [ ]:
# --- Limpieza: construcción de una versión depurada del dataset ---
df_limpio = df.copy()
df_limpio["texto_usuario"] = df_limpio["texto_usuario"].replace("", np.nan)
df_limpio = df_limpio.dropna(subset=["texto_usuario"])
df_limpio = df_limpio.drop_duplicates(subset=["texto_usuario"])

print(f"Registros originales:          {df.shape[0]}")
print(f"Registros después de limpieza: {df_limpio.shape[0]}")
print(f"Registros eliminados:          {df.shape[0] - df_limpio.shape[0]}")


### 4.3 Ruido gramatical y outliers léxicos

Se usa un **diagrama de caja (boxplot)** para detectar mensajes atípicamente
largos o cortos por nivel de lenguaje e intención, se calculan outliers mediante
el método del **rango intercuartílico (IQR)**, y se aplica una heurística simple
para caracterizar mensajes con posibles **incoherencias ortográficas o
sintácticas simuladas** (letras duplicadas consecutivas, típico artefacto
introducido en la Sección 1.5).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_limpio, x="nivel_lenguaje", y="longitud_char",
            hue="nivel_lenguaje", palette="Set2", legend=False, ax=axes[0])
axes[0].set_title("Outliers de longitud (caracteres) por nivel de lenguaje")

sns.boxplot(data=df_limpio, x="intencion", y="conteo_palabras",
            hue="intencion", palette="Set3", legend=False, ax=axes[1])
axes[1].set_title("Outliers de longitud (palabras) por intención")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


In [ ]:
# Outliers por rango intercuartílico (IQR) sobre longitud en caracteres
Q1 = df_limpio["longitud_char"].quantile(0.25)
Q3 = df_limpio["longitud_char"].quantile(0.75)
IQR = Q3 - Q1
lim_inferior = Q1 - 1.5 * IQR
lim_superior = Q3 + 1.5 * IQR

outliers = df_limpio[(df_limpio["longitud_char"] < lim_inferior) |
                      (df_limpio["longitud_char"] > lim_superior)]

print(f"Límite inferior: {lim_inferior:.1f} caracteres")
print(f"Límite superior: {lim_superior:.1f} caracteres")
print(f"Número de mensajes atípicos (outliers): {outliers.shape[0]}")

outliers[["texto_usuario", "intencion", "nivel_lenguaje", "longitud_char"]]


In [ ]:
# Heurística de detección de posibles incoherencias ortográficas/gramaticales:
# letras consecutivas repetidas dentro de una misma palabra (p. ej. "cciita").
# NOTA: es una heurística simple con fines ilustrativos; puede generar falsos
# positivos en palabras con dobles letras legítimas (p. ej. "carro").
patron_letra_duplicada = re.compile(r"(\w)\1{1,}")

def contiene_posible_ruido(texto):
    if pd.isna(texto):
        return False
    return bool(patron_letra_duplicada.search(texto.lower()))

df_limpio["posible_ruido"] = df_limpio["texto_usuario"].apply(contiene_posible_ruido)

print("Registros con posible ruido ortográfico/gramatical simulado:")
print(df_limpio["posible_ruido"].value_counts())

df_limpio.loc[df_limpio["posible_ruido"], ["texto_usuario", "intencion", "nivel_lenguaje"]].head(10)


## 5. Conclusiones del EDA

- Se integró con éxito el **léxico real español–wayuunaiki** del repositorio
  `Manuel-2011/RLTranslator` al módulo de generación de mensajes en
  `wayuunaiki_basico`, evidenciando además una **brecha léxica real**: términos
  institucionales como *EPS*, *autorización* o *reclamo* no cuentan con
  traducción directa en el diccionario público, lo que confirma la necesidad de
  un enfoque de *code-switching* (y no de traducción automática completa) para
  esta primera versión del chatbot.
- El dataset sintético refleja **desbalance de clases** tanto en `intencion`
  (`solicitar_cita` sobre-representada) como en `nivel_lenguaje`
  (`wayuunaiki_basico` sub-representado), lo que deberá abordarse en el
  modelado (p. ej. con *re-sampling*, ponderación de clases o aumento de datos
  específico para wayuunaiki básico).
- Los mensajes en **lenguaje formal** tienden a ser más largos (más caracteres
  y palabras) que los de **lenguaje simplificado** y **wayuunaiki básico**, lo
  cual es consistente con el propósito de accesibilidad del proyecto.
- Se identificaron y cuantificaron **valores nulos, vacíos y duplicados**, y se
  generó una versión limpia (`df_limpio`) apta para las siguientes etapas de
  preprocesamiento y modelado NLP.
- La detección de **outliers léxicos** y de **ruido ortográfico/gramatical
  simulado** evidencia la necesidad de un pipeline robusto de normalización de
  texto (corrección ortográfica, normalización de tildes, manejo de
  *code-switching*) antes del entrenamiento del clasificador de intenciones y
  del extractor de entidades del chatbot.
